# 03c · Generation benchmark (Gemini + Groq)

| ID | System | Retrieval | LLM | Prompt |
|---|---|---|---|---|
| L0 | LLM, no retrieval | none | GENERATOR (Gemini) | PlantIQ's prompt with no passages |
| SK | Selva Kumar et al. | LangChain dense top-4 | BASELINE_LLM (Groq Llama-3.1-8B, standing in for their GPT-3.5) | LangChain's default QA prompt |
| PQ | PlantIQ | full hybrid (top-3) | GENERATOR (Gemini) | PlantIQ's prompt, language router, dose audit |

Every system gets the same question and disease label, temperature 0.2. One JUDGE call per answer (Gemini, a
different and stronger model than the generator) gives the RAGAS-definition metrics: faithfulness, answer
relevancy, context precision, answer correctness, plus abstention. Unsupported doses come from PlantIQ's dose
audit applied to every system's answer against the passages it was given (no LLM needed).

Calls: about 3 x 85 answers + 85 router calls + 255 judge calls. All cached in `03_rag/llm_cache/`: if a
free-tier quota runs out the notebook stops with a message; just run it again later and it continues.

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd()
while not (ROOT / "PLAN.md").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
sys.path.insert(0, str(ROOT / "03_rag"))

import os
import time

import numpy as np
import pandas as pd
from tqdm.auto import tqdm

from common import config, env
from common import rag_eval as R
from common.llm import LLM, load_env, role_model
from plantiq_rag_snapshot.advisory import SYSTEM_PROMPT, LLMAdapter, build_prompt, display_name, generate_advisory
from plantiq_rag_snapshot.build import build_retriever, load_embedder
from plantiq_rag_snapshot.profiles import BALANCED, QUALITY
from plantiq_rag_snapshot.query import audit_dosages
from plantiq_rag_snapshot.text import has_kannada

config.ensure_dirs()
load_env()
device = env.require_cuda().type
qs = pd.read_json(config.RAG_DATA / "questions.jsonl", lines=True)
print(len(qs), "questions,", int(qs.out_of_kb.sum()), "out of the documents")

PROFILE = BALANCED if config.SMOKE else QUALITY
SK_EMB = "hf:sentence-transformers/all-MiniLM-L6-v2" if config.SMOKE else \
    os.getenv("SK_EMBEDDINGS", "openai:text-embedding-3-small" if os.getenv("OPENAI_API_KEY", "").strip() else
              "hf:sentence-transformers/all-mpnet-base-v2")
GEN, BASE, JUDGE = (role_model(r, verbose=True) for r in ("GENERATOR", "BASELINE_LLM", "JUDGE"))
print("generator:", GEN, "| baseline LLM:", BASE, "| judge:", JUDGE, "| SK embeddings:", SK_EMB)
gen_llm, base_llm, judge = LLM(GEN), LLM(BASE), LLM(JUDGE)

In [ ]:
embedder = load_embedder(PROFILE, device)
rag = build_retriever(PROFILE, config.KB_DIR, config.RAG_INDEX / f"pq_{PROFILE.name}", device=device,
                      embedder=embedder)
sk = R.SKRetriever(SK_EMB, config.KB_DIR, config.RAG_INDEX, device=device)

## Generate answers

In [ ]:
def as_docs(hits):
    return [{"text": h["text"], "source": h["source"], "metadata": {"page": h["page"]}} for h in hits]


answers = []
for _, q in tqdm(qs.iterrows(), total=len(qs), desc="generate"):
    target = "kn" if q["lang"] == "kn" or has_kannada(q["question"]) else "en"
    label = display_name(q["disease"])
    # L0: no retrieval
    r = gen_llm.generate(build_prompt(q["disease"], 100.0, None, q["question"], target, [], False),
                         system=SYSTEM_PROMPT, temperature=0.2, max_tokens=900)
    answers.append({"id": q["id"], "system": "L0", "answer": r.text, "contexts": [], "docs": [],
                    "latency_s": r.latency_s})
    # SK: LangChain dense top-4 + baseline LLM with LangChain's QA prompt
    t0 = time.perf_counter()
    hits = sk.search(f"{label}: {q['question']}", k=4)
    t_ret = time.perf_counter() - t0
    ctx = [h["text"] for h in hits]
    r = base_llm.generate(R.SK_PROMPT.format(context="\n\n".join(ctx), question=f"{label}: {q['question']}"),
                          temperature=0.2, max_tokens=900)
    answers.append({"id": q["id"], "system": "SK", "answer": r.text, "contexts": ctx, "docs": as_docs(hits),
                    "latency_s": t_ret + r.latency_s})
    # PQ: the full PlantIQ advisory path
    adapter = LLMAdapter(gen_llm)
    rag._cache.clear()
    t0 = time.perf_counter()
    adv = generate_advisory(disease=q["disease"], confidence=100.0, env=None, question=q["question"],
                            language=q["lang"], rag=rag, llm=adapter)
    wall = time.perf_counter() - t0
    llm_live = sum(c.latency_s for c in adapter.calls if not c.cached)
    llm_rec = sum(c.latency_s for c in adapter.calls)
    answers.append({"id": q["id"], "system": "PQ", "answer": adv.answer, "contexts": [d["text"] for d in adv.docs],
                    "docs": adv.docs, "latency_s": wall - llm_live + llm_rec, "blended": adv.is_blended})
ans = pd.DataFrame(answers).merge(qs, on="id")
print(len(ans), "answers")

## Score answers (one judge call each)

In [ ]:
scores = []
for _, a in tqdm(ans.iterrows(), total=len(ans), desc="judge"):
    row = {"id": a["id"], "system": a["system"], "topic": a["topic"], "lang": a["lang"], "out_of_kb": a["out_of_kb"],
           "latency_s": a["latency_s"]}
    audit = audit_dosages(a["answer"], a["docs"])
    row["doses_in_answer"] = len(audit["detected_dosages"])
    row["unsupported_doses"] = len(audit["unverified_dosages"])
    row["has_unsupported_dose"] = int(bool(audit["unverified_dosages"]))
    row.update(R.judge_answer(judge, embedder, a["question"], a["gold_answer"], a["answer"], a["contexts"],
                              has_context=a["system"] != "L0"))
    scores.append(row)
sc = pd.DataFrame(scores)
ans.drop(columns=["docs"]).to_json(config.RAG_RESULTS / "generation_answers.jsonl", orient="records", lines=True,
                                   force_ascii=False)
sc.to_csv(config.RAG_RESULTS / "generation_rows.csv", index=False)

## Summary

In [ ]:
SYS = {"PQ": "PlantIQ hybrid RAG", "SK": f"Selva Kumar et al. (dense RAG + {BASE.split(':', 1)[1]})",
       "L0": "LLM, no retrieval"}
inkb, ook = sc[sc.out_of_kb == 0], sc[sc.out_of_kb == 1]
M_ = ["faithfulness", "answer_relevancy", "context_precision", "answer_correctness"]
summary = {}
for s in SYS:
    d, o = inkb[inkb.system == s], ook[ook.system == s]
    summary[s] = {"name": SYS[s], **{m: R.bootstrap_mean_ci(d[m]) for m in M_},
                  "unsupported_dose_rate": float(sc[sc.system == s]["has_unsupported_dose"].mean()),
                  "answers_with_doses": float((sc[sc.system == s]["doses_in_answer"] > 0).mean()),
                  "abstention_out_of_kb": float(o["abstained"].mean()) if len(o) else float("nan"),
                  "false_abstention_in_kb": float(d["abstained"].mean()),
                  "latency_median_s": float(sc[sc.system == s]["latency_s"].median()),
                  "latency_p95_s": float(sc[sc.system == s]["latency_s"].quantile(0.95)),
                  "by_lang": {l: {m: float(d.loc[d.lang == l, m].mean()) for m in M_} for l in ("en", "kn")}}
pq_ids = inkb[inkb.system == "PQ"]["id"]
tests = {s: {m: R.wilcoxon_paired(inkb[inkb.system == "PQ"].set_index("id")[m],
                                  inkb[inkb.system == s].set_index("id").reindex(pq_ids)[m])
             for m in M_} for s in SYS if s != "PQ"}
env.write_json(config.RAG_RESULTS / "generation.json", {
    "systems": summary, "wilcoxon_pq_vs": tests, "models": {"generator": GEN, "baseline": BASE, "judge": JUDGE},
    "judge_mode": "single call per answer, RAGAS metric definitions", "sk_embeddings": SK_EMB,
    "profile": PROFILE.name, "n_questions": int(len(qs)), "meta": env.run_meta()})
pd.DataFrame({SYS[s]: {**{m: v[m]["mean"] for m in M_}, "unsupported doses": v["unsupported_dose_rate"],
                       "abstain (outside docs)": v["abstention_out_of_kb"], "latency p50 s": v["latency_median_s"]}
              for s, v in summary.items()}).T